# CyberGuard AI — Phishing URL Detection

## Project Overview

This notebook builds a machine-learning model to classify web pages as **phishing** or **legitimate**.

The workflow covers:

1. Loading and inspecting the dataset
2. Checking data quality
3. Exploring the target distribution
4. Preparing features and labels
5. Training a Random Forest classifier
6. Evaluating model performance
7. Visualizing the confusion matrix, ROC curve, and feature importance
8. Saving the trained model and feature information
9. Testing predictions on unseen samples

> **Note:** The model uses the engineered numerical features already provided in the dataset. The raw `url` column is not used directly for training.

## 1. Import Required Libraries

In [ ]:
import json
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

## 2. Load the Dataset

In [ ]:
DATA_PATH = "../Dataset/dataset_phishing.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Inspect the Dataset

In [ ]:
df.info()

### Check for Missing Values

In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

### Summary Statistics

In [ ]:
df.describe()

## 4. Explore the Target Variable

In [ ]:
status_counts = df["status"].value_counts()
status_counts

In [ ]:
phishing_ratio = (df["status"] == "phishing").mean()

print(f"Phishing URLs: {phishing_ratio:.1%}")
print(f"Legitimate URLs: {1 - phishing_ratio:.1%}")

In [ ]:
plt.figure(figsize=(6, 4))
status_counts.plot(kind="bar")
plt.title("Phishing vs Legitimate URLs")
plt.xlabel("URL Status")
plt.ylabel("Number of URLs")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Prepare Labels and Features

The original `status` column is converted into a binary target:

- `0` → Legitimate
- `1` → Phishing

The raw `url` column is excluded because the dataset already contains engineered features for classification.

In [ ]:
df["label"] = (df["status"] == "phishing").astype(int)

X = df.drop(columns=["status", "label", "url"], errors="ignore")
y = df["label"]

print(f"Number of samples: {len(X):,}")
print(f"Number of features: {X.shape[1]}")

In [ ]:
pd.crosstab(df["status"], df["label"])

## 6. Split the Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")

## 7. Train the Random Forest Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print("Model training completed.")

## 8. Generate Predictions

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

y_pred[:10]

## 9. Evaluate Model Performance

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Legitimate", "Phishing"],
    )
)

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Legitimate", "Phishing"],
    yticklabels=["Legitimate", "Phishing"],
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

### ROC Curve and AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"ROC-AUC: {roc_auc:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Feature Importance

In [ ]:
feature_importance = (
    pd.Series(model.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(8, 5))
feature_importance.sort_values().plot(kind="barh")
plt.title("Top 15 Features for Phishing Detection")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## 10. Test the Model with Sample Records

The examples below use records from the held-out test set. This verifies that the prediction function returns both the predicted class and the estimated phishing probability.

In [ ]:
def predict_from_features(features: dict):
    """Predict whether a feature dictionary represents a phishing URL."""
    sample = pd.DataFrame([features])
    sample = sample.reindex(columns=X.columns, fill_value=0)

    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0, 1]

    label = "PHISHING" if prediction == 1 else "LEGITIMATE"

    print(f"Prediction: {label}")
    print(f"Phishing probability: {probability:.1%}")

    return label, probability

In [ ]:
phishing_sample = X_test[y_test == 1].iloc[0].to_dict()
predict_from_features(phishing_sample)

In [ ]:
legitimate_sample = X_test[y_test == 0].iloc[0].to_dict()
predict_from_features(legitimate_sample)

## 11. Save the Trained Model

In [ ]:
MODEL_PATH = "phishing_model.pkl"
FEATURES_PATH = "model_columns.pkl"
INFO_PATH = "model_info.json"

joblib.dump(model, MODEL_PATH)
joblib.dump(list(X.columns), FEATURES_PATH)

model_info = {
    "model_type": "RandomForestClassifier",
    "n_estimators": 100,
    "max_depth": 15,
    "dataset": "Web Page Phishing Detection",
    "total_samples": int(len(df)),
    "training_samples": int(len(X_train)),
    "testing_samples": int(len(X_test)),
    "feature_count": int(X.shape[1]),
    "accuracy": float(accuracy),
    "roc_auc": float(roc_auc),
}

with open(INFO_PATH, "w", encoding="utf-8") as file:
    json.dump(model_info, file, indent=2)

print("Saved files:")
print(f"- {MODEL_PATH}")
print(f"- {FEATURES_PATH}")
print(f"- {INFO_PATH}")

## 12. Verify Saved Files

In [ ]:
for file_path in [MODEL_PATH, FEATURES_PATH, INFO_PATH]:
    size_kb = os.path.getsize(file_path) / 1024
    print(f"{file_path}: {size_kb:.1f} KB")

## 13. Final Model Summary

The notebook trains a **Random Forest classifier** for phishing URL detection using the engineered features in the dataset.

### Model Configuration

- **Algorithm:** Random Forest
- **Trees:** 100
- **Maximum depth:** 15
- **Train/test split:** 80% / 20%
- **Random state:** 42
- **Stratified split:** Yes

### Saved Artifacts

- `phishing_model.pkl` — trained Random Forest model
- `model_columns.pkl` — feature names used during training
- `model_info.json` — model configuration and evaluation metrics

These files can be loaded by a separate application or API for inference.